# clearSKY AI: 2,000-Scene Dataset + All-Model Training

Run this notebook in Google Colab with a GPU runtime. It builds the 2,000-scene fusion dataset on Google Drive, then trains U-Net, Attention U-Net, Swin-UNet, and multi-sensor fusion.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone Or Update Repo

In [ ]:
%%bash
cd /content
if [ ! -d clearSKY_AI ]; then
  git clone https://github.com/Vishwajitsingh-rajput-27/clearSKY_AI.git
else
  cd clearSKY_AI && git pull
fi

## 3. Install Dependencies

In [ ]:
%%bash
cd /content/clearSKY_AI/apps/api
pip install -r requirements.txt
pip install -r requirements-dataset.txt

## 4. Authenticate Earth Engine

Replace `YOUR_GOOGLE_CLOUD_PROJECT` with your Google Cloud project id. If your Earth Engine account does not require a project, remove the `project=...` argument.

In [ ]:
import ee
ee.Authenticate()
ee.Initialize(project='YOUR_GOOGLE_CLOUD_PROJECT')

## 5. Dry-Run The 2,000-Scene Search

In [ ]:
%%bash
cd /content/clearSKY_AI/apps/api
python -m app.ai.datasets.build_fusion_100 \
  --config configs/fusion_dataset_2000_colab.json \
  --project YOUR_GOOGLE_CLOUD_PROJECT \
  --dry-run

## 6. Build A Small 20-Scene Trial

Run this before the full 2,000-scene job.

In [ ]:
%%bash
cd /content/clearSKY_AI/apps/api
python -m app.ai.datasets.build_fusion_100 \
  --config configs/fusion_dataset_2000_colab.json \
  --project YOUR_GOOGLE_CLOUD_PROJECT \
  --target-count 20

## 7. Build Full 2,000 Scenes

This writes to `/content/drive/MyDrive/clearSKY_AI/data`. If Colab disconnects, rerun the same cell. Completed scenes are skipped.

In [ ]:
%%bash
cd /content/clearSKY_AI/apps/api
python -m app.ai.datasets.build_fusion_100 \
  --config configs/fusion_dataset_2000_colab.json \
  --project YOUR_GOOGLE_CLOUD_PROJECT

## 8. Quick One-Epoch All-Model Test

In [ ]:
%%bash
cd /content/clearSKY_AI/apps/api
python -m app.ai.training.train_all \
  --device auto \
  --epochs 1 \
  --train-manifest /content/drive/MyDrive/clearSKY_AI/data/manifests/train.json \
  --val-manifest /content/drive/MyDrive/clearSKY_AI/data/manifests/val.json \
  --checkpoint-dir /content/drive/MyDrive/clearSKY_AI/models/checkpoints \
  --summary-path /content/drive/MyDrive/clearSKY_AI/models/checkpoints/train_all_smoke_summary.json

## 9. Full All-Model Training

In [ ]:
%%bash
cd /content/clearSKY_AI/apps/api
python -m app.ai.training.train_all \
  --device auto \
  --train-manifest /content/drive/MyDrive/clearSKY_AI/data/manifests/train.json \
  --val-manifest /content/drive/MyDrive/clearSKY_AI/data/manifests/val.json \
  --checkpoint-dir /content/drive/MyDrive/clearSKY_AI/models/checkpoints \
  --summary-path /content/drive/MyDrive/clearSKY_AI/models/checkpoints/train_all_summary.json

## 10. Inspect Outputs

In [ ]:
%%bash
find /content/drive/MyDrive/clearSKY_AI/models/checkpoints -maxdepth 3 -type f | sort | sed -n '1,80p'